# Notebook 14 — The gene panel: two coupled networks tested for selection across primates

**Scope.** This notebook defines and justifies the gene panel used in the primate sexual-dichromatism
selection scan. Sexual dichromatism in hair/pelage is a pigmentation phenotype expressed in a sex-limited way,
so we test two coupled molecular systems for signatures of selection across the primate phylogeny: the
**melanocyte pigmentation network** and the **sex-hormone (hypothalamic-pituitary-gonadal) axis**. For every
gene in the panel we record the evidence that places it in its module, and we screen every gene for clean
one-to-one orthology across primates — a prerequisite for a trustworthy dN/dS analysis.

The final panel is **110 genes: 57 pigmentation and 53 sex-hormone**, an intentionally balanced design so that
per-module comparisons are not driven by how many genes each module happens to contain.

**Evidence sources for module membership (all internal to this project's curated network).**
- **OMIM pigmentation phenotype** — the gene, when disrupted in humans, causes a documented pigmentation
  disorder (oculocutaneous albinism, Hermansky-Pudlak, Griscelli, Chediak-Higashi, vitiligo, or a
  hyper-/hypopigmentation phenotype).
- **Melanogenesis mechanism** — membership in the curated melanin-synthesis / melanocyte-signaling pathway.
- **Baxter et al. 2019** — a curated reference set of pigmentation genes (*Pigment Cell Melanoma Res* 32:348,
  doi:10.1111/pcmr.12743; 657 human gene symbols).
- **Melanosome proteome** (mass-spectrometry of purified melanosomes) and **functional CRISPR screen**
  (corroborating layers).
- **Endocrine-pathway position** — for the hormone module, curated membership in steroid biosynthesis, the HPG
  axis, sex-steroid receptors, or their coactivators/carriers.

In [ ]:
import os, pandas as pd
REPO = os.environ.get("PIGNET_REPO", os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")))
CG   = os.path.join(REPO, "comparative-genomics")
DATA = os.path.join(CG, "analysis", "module_selection", "data")
J = pd.read_csv(os.path.join(DATA, "nb14_panel_justification.csv"))
# The analytic panel: hormone axis + pigmentation genes that pass the orthology screen.
J["in_panel"] = (J.status == "already_run") | (J.ortholog_risk == "clean")
panel = J[J.in_panel].copy()
print(f"ANALYTIC PANEL: {len(panel)} genes")
print("  by module:", panel.module.value_counts().to_dict())
npig = (panel.module=="pigmentation").sum(); nhor = (panel.module=="hormone").sum()
print(f"  module balance-neutral point: {2*npig/(npig+nhor)-1:+.3f}  (0 = perfectly balanced panel)")

## 1 — The pigmentation module (57 genes)

The pigmentation module spans the melanocyte from signal reception to melanosome delivery, in five functional
layers:

- **Receptor signaling** — MC1R/ASIP/POMC melanocortin signaling and the endothelin (EDN3/EDNRB) and KIT/KITLG
  axes that set melanocyte fate and the eumelanin/pheomelanin switch.
- **Transcription** — the MITF/SOX10/PAX3/TFAP2A/LEF1 regulatory core.
- **Melanogenesis enzymes** — TYR, TYRP1, DCT (melanin synthesis) and the MAPK/Wnt/endothelin signaling arm
  that regulates MITF (CTNNB1, HRAS, RAF1, MAPK3, EGFR, HGF, CDC42, SRC, and related regulators).
- **Melanosome structure & transport** — PMEL, MLANA, the OCA/SLC transporters, and the melanosome-biogenesis
  machinery: the Hermansky-Pudlak (HPS1/3/4/5, AP-3, BLOC-1) and Griscelli (RAB27A, MYO5A) systems, LYST, and
  ocular-albinism GPR143.

Every gene carries at least one pigmentation-specific evidence source (below).

In [ ]:
pig = panel[panel.module == "pigmentation"].copy()
order = ["receptor_signaling","transcription","enzyme","melanogenesis_regulation",
         "melanosome","melanosome_transport","melanosome_biogenesis","regulatory"]
pig["category"] = pd.Categorical(pig.category, categories=order, ordered=True)
for cat in order:
    sub = pig[pig.category == cat]
    if len(sub):
        print(f"\n{cat} ({len(sub)}): " + ", ".join(sorted(sub.gene)))

## 2 — The sex-hormone module (53 genes)

The hormone module represents the full hypothalamic-pituitary-gonadal axis and sex-steroid metabolism, because
the a-priori hypothesis is that sex-steroid signaling gates the sex-limited *expression* of the pigmentation
phenotype. It spans steroid biosynthesis, the HPG axis (GnRH/gonadotropins), the androgen and estrogen receptor
axes, GnRH signaling, and the coactivators/corepressors/carriers that modulate receptor activity.

In [ ]:
hor = panel[panel.module == "hormone"].copy()
for cat, sub in hor.groupby("category"):
    print(f"{cat} ({len(sub)}): " + ", ".join(sorted(sub.gene)))

## 3 — Orthology screen

A dN/dS scan is trustworthy only when the gene has a clean **one-to-one ortholog** in each primate genome. Two
failure modes threaten this: (i) **paralog mis-mapping**, where the ortholog-extraction step lands on a close
paralog and corrupts the alignment (a risk for members of large gene families); and (ii) **incomplete ortholog
coverage**, where short or poorly annotated genes are not recovered in scaffold-level assemblies or the deeper
lineages. Every candidate pigmentation gene was screened against Ensembl Compara (homology across 12 reference
primates; human within-species paralog count). Genes that failed the screen were excluded from the analytic
panel and are recorded as such — an auditable exclusion, not a silent drop.

In [ ]:
excluded = J[~J.in_panel]
print(f"genes excluded for orthology risk: {len(excluded)}")
print(excluded[["gene","category","n_primate_orth","n_within_paralog","ortholog_risk"]].to_string(index=False))

## 4 — Panel balance

The two modules are deliberately matched in breadth (57 vs 53) so that a per-module comparison reflects biology
rather than sampling. Module-level selection should still be reported as a **per-gene rate**
(genes-under-selection / genes-tested, within each module) rather than a raw count, so it is robust to the
exact panel size.

In [ ]:
print(f"pigmentation module: {npig} genes")
print(f"sex-hormone module:  {nhor} genes")
print(f"balance-neutral point: {2*npig/(npig+nhor)-1:+.3f}  (near 0 => a balanced panel)")

## Result

A documented, orthology-screened panel of **110 genes** (57 pigmentation, 53 sex-hormone), each traceable to a
module-membership evidence source, with primate orthology verified. This is the gene set carried into the
per-branch (aBSREL) and per-origin (RELAX) selection analyses. The full annotated table, including the internal
provenance and orthology columns, is `data/nb14_panel_justification.csv`.

---
*Internal note (not part of the manuscript panel description): the `status` column of the justification table
marks which genes were in the original scan (`already_run`) versus added in the melanosome/regulation expansion
(`to_run`), and the `ortholog_risk` column records the 13 candidates excluded by the orthology screen. These
are for provenance tracking only.*

In [ ]:
internal = J.groupby(["module","status"]).size().unstack(fill_value=0)
print("internal provenance (status x module):")
print(internal.to_string())